# 🛡️ Glu-Stock: 03_EXECUTION_ENGINE
**Phase**: Risk Management & Trade Execution

This notebook retrieves signals from the Firebase `signals` queue, calculates position sizes (ATR/Kelly), and updates the global `trades` state in the cloud.

In [ ]:
# 📦 SECTION 1: INSTALLATION
!pip install -q yfinance firebase-admin pandas

In [ ]:
# 🏗️ SECTION 2: INFRASTRUCTURE (Firebase & Secrets)
import json, os, firebase_admin, pandas as pd, yfinance as yf
from firebase_admin import credentials, db
from datetime import datetime
from typing import List, Dict, Any, Optional
from kaggle_secrets import UserSecretsClient

class KaggleInfra:
    @staticmethod
    def load_secrets():
        user_secrets = UserSecretsClient()
        return {
            "url": user_secrets.get_secret("FIREBASE_URL"),
            "key": json.loads(user_secrets.get_secret("FIREBASE_KEY_JSON"))
        }

class FirebaseHandler:
    def __init__(self, secrets):
        if not firebase_admin._apps:
            cred = credentials.Certificate(secrets['key'])
            firebase_admin.initialize_app(cred, {'databaseURL': secrets['url']})
        self.root_ref = db.reference("glu_stock")

    def get_and_clear_queue(self, queue_name: str) -> List[Any]:
        ref = self.root_ref.child(f"task_queue/{queue_name}")
        tasks = ref.get()
        if not tasks: return []
        ref.delete()
        return list(tasks.values())
        
    def insert_trade(self, trade_data: Dict[str, Any]):
        self.root_ref.child("trades").push(trade_data)
        
    def log_event(self, phase, details):
        self.root_ref.child("history").push({'timestamp': datetime.now().isoformat(), 'phase': phase.upper(), 'details': details})

In [ ]:
# 🧠 SECTION 3: CORE LOGIC (Risk & Execution)
class RiskManager:
    def calculate_size(self, price, conviction, total_equity=100000000):
        # Simple ATR-like sizing demo
        return int((total_equity * 0.02 * conviction) / price)

class TradingAgent:
    def __init__(self, fb):
        self.fb = fb
        self.risk = RiskManager()

    def execute_signals(self, signals):
        for ticker, data in signals.items():
            print(f"🚀 Executing trade for {ticker}...")
            shares = self.risk.calculate_size(data['price'], data['conviction'])
            if shares > 0:
                trade_data = {
                    'ticker': ticker,
                    'shares': shares,
                    'entry_price': data['price'],
                    'entry_date': datetime.now().isoformat(),
                    'conviction': data['conviction'],
                    'status': 'OPEN'
                }
                self.fb.insert_trade(trade_data)
                self.fb.log_event("EXECUTION", f"Opened {ticker} @ {data['price']} ({shares} shares)")

In [ ]:
# 🚀 SECTION 4: MAIN EXECUTION
def run_execution():
    secrets = KaggleInfra.load_secrets()
    fb = FirebaseHandler(secrets)
    
    # 1. Pull signals queue
    signal_batches = fb.get_and_clear_queue("signals")
    if not signal_batches: print("📭 Queue empty."); return
    
    all_signals = {}
    for batch in signal_batches: all_signals.update(batch)

    # 2. Execute
    agent = TradingAgent(fb)
    agent.execute_signals(all_signals)
    print("✅ Execution complete.")

run_execution()